# Tokenizer Comparison
This notebook is for comparing tokenizers across **tiktoken**, **Hugging Face tokenizers**, and **SentencePiece-family tokenizers**.


By the end of this notebook, students should be able to:

- Explain why tokenization matters for LLM quality, efficiency, and cost.
- Compare tokenizer behavior across English, code, emoji, whitespace, and multilingual text.
- Interpret token pieces, token IDs, round-trip decoding, and token counts.
- See how the same string can become very different token sequences across libraries.

## Why tokenizers matter

A tokenizer is the bridge between raw text and model inputs. Models do not read characters or words directly; they read token IDs.

This means tokenization affects:

- Context window usage.
- API cost for token-billed systems.
- How efficiently code, rare words, emoji, and multilingual text are represented.
- Whether a model splits text into intuitive subwords or awkward byte-like fragments.

**same text, different tokenizer, different computation budget**.


In [1]:
!pip -q install tiktoken transformers sentencepiece pandas numpy matplotlib seaborn ipywidgets


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 13.0 MB/s eta 0:00:00


We use:

- `tiktoken` for OpenAI-style encodings.
- `transformers` for Hugging Face tokenizers.
- `sentencepiece` indirectly through tokenizers such as T5/LLaMA-family tokenizers.

In [2]:
import time
import math
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tiktoken
from transformers import AutoTokenizer

pd.set_option('display.max_colwidth', 200)
sns.set_theme(style='whitegrid')


## Sample texts

The sample set is intentionally diverse:

- Simple English.
- Accented text.
- Code.
- Hindi mixed with English.
- Emoji.
- Repeated whitespace.
- Long compound words.
- Structured prompts.

In [3]:
examples = [
    'Hello world',
    'Café 🚀',
    'def add(x, y): return x + y',
    'नमस्ते world',
    'The cat sat on the mat.',
    'Email me at demo@example.com by 10:30 AM.',
    'GPU-memory-efficient attention is important for long-context models.',
    '🙂🙂🙂 emoji-rich text with symbols ©™✓',
    'Multiple     spaces and new lines	too.',
    'Translate this sentence from English to Hindi: The weather is pleasant today.'
]

examples


['Hello world',
 'Café 🚀',
 'def add(x, y): return x + y',
 'नमस्ते world',
 'The cat sat on the mat.',
 'Email me at demo@example.com by 10:30 AM.',
 'GPU-memory-efficient attention is important for long-context models.',
 '🙂🙂🙂 emoji-rich text with symbols ©™✓',
 'Multiple     spaces and new lines\ttoo.',
 'Translate this sentence from English to Hindi: The weather is pleasant today.']

## Load tokenizers

This notebook compares multiple tokenizer families:

- `tiktoken:r50k_base` — older GPT-style BPE.
- `tiktoken:cl100k_base` — newer GPT tokenizer family.
- `tiktoken:o200k_base` — larger, newer vocabulary family.
- `hf:gpt2` — Hugging Face GPT-2 tokenizer.
- `hf:llama` — SentencePiece-style tokenizer behavior via a LLaMA-family tokenizer.
- `hf:t5-small` — SentencePiece tokenizer from the T5 family.


They are the same concept (text → token IDs using BPE), but different libraries — tiktoken is a minimal, fast encoder for OpenAI models only, while HuggingFace tokenizer is a full-featured preprocessing toolkit that supports all major algorithms and all major model families.

tiktoken does one thing only — fast tokenization. No model loading, no attention masks, no padding logic.

HuggingFace tokenizer is a full preprocessing pipeline — it handles padding, truncation, attention masks, token type IDs, batch encoding, and chat templates.



In [4]:
#cl100k_base is technically an encoding in tiktoken's vocabulary, but it is the tokenizer — it defines the full tokenization behaviour.
encodings = {
    'tiktoken:r50k_base': tiktoken.get_encoding('r50k_base'),
    'tiktoken:cl100k_base': tiktoken.get_encoding('cl100k_base'), #used by GPT-3.5 and GPT-4 family, vocabulary of ~100,277 tokens
    'tiktoken:o200k_base': tiktoken.get_encoding('o200k_base'), #used by GPT-4o,  vocab of ~200,019 tokens — more efficient on code and multilingual text
}

hf_tokenizers = {
    'hf:gpt2': AutoTokenizer.from_pretrained('gpt2'), #BPE, vocab size 50,257, marks spaces with Ġ (special G character)
    'hf:llama': AutoTokenizer.from_pretrained('NousResearch/Llama-2-7b-chat-hf'),#SentencePiece, vocab size 32,000, marks spaces with ▁
    'hf:t5-small': AutoTokenizer.from_pretrained('t5-small'),#WordPiece, vocab size 30,522, marks continuation subwords with ##
}

list(encodings.keys()) + list(hf_tokenizers.keys())


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

['tiktoken:r50k_base',
 'tiktoken:cl100k_base',
 'tiktoken:o200k_base',
 'hf:gpt2',
 'hf:llama',
 'hf:t5-small']

## Helper functions

These functions standardize the comparison across libraries.

They return:

- Token pieces.
- Token IDs.
- Decoded text.
- Token count.
- Whether encode-decode gives the original string back.


In [5]:
#Tokenize any string using a **tiktoken** encoding and return a structured
#dictionary with pieces, IDs, decoded text, token count, and a round-trip check.

#tiktoken does NOT have a `convert_ids_to_tokens()` method like HuggingFace.The only way to see what each individual token looks like
#as human-readable text is to **decode each ID separately**

def tiktoken_view(enc, text):
    """
    Tokenize `text` using a tiktoken encoding and return a structured dict.

    Parameters
    ----------
    enc  : tiktoken.Encoding
           A loaded encoding object, e.g. tiktoken.get_encoding('cl100k_base')
    text : str
           Any raw input string — English, code, Hindi, emoji, etc.

    Returns
    -------
    dict with keys: pieces, ids, decoded, token_count, roundtrip_match
    """

    # enc.encode() applies BPE merge rules to split text into subwords, then maps each subword to its integer ID in the vocabulary
    # Example: "Hello world" → [9906, 1917]
    ids = enc.encode(text)

    #  Decode each ID individually to get human-readable pieces
    # Example: 9906 → "Hello",  1917 → " world"   (notice the leading space)
    pieces = [enc.decode([i]) for i in ids]

    # Decode the full ID list back into a single string
    decoded = enc.decode(ids)

   # len(ids) gives the number of tokens.
    # Fewer tokens = more efficient = less context window used = lower API cost
    token_count = len(ids)

    # If decoded == text, the tokenizer perfectly reconstructed the original.
    # tiktoken almost always passes this check.
    # Some HuggingFace tokenizers fail for: leading spaces, accents, rare Unicode.
    roundtrip_match = (decoded == text)

    return {
        'pieces':        pieces,
        'ids':           ids,
        'decoded':       decoded,
        'token_count':   token_count,
        'roundtrip_match': roundtrip_match,
    }

Same job as tiktoken_view but uses the HuggingFace Transformers API, which is the standard API for BERT, GPT-2, LLaMA, T5, Mistral, Qwen, etc.

In [6]:
def hf_view(tok, text):
    """
    Tokenize `text` using any HuggingFace tokenizer and return a structured dict.

    Parameters
    ----------
    tok  : PreTrainedTokenizer or PreTrainedTokenizerFast
           Any HuggingFace tokenizer (BPE, WordPiece, SentencePiece — all work)
    text : str
           Any raw input string

    Returns
    -------
    dict with keys: pieces, ids, decoded, token_count, roundtrip_match
    Same format as tiktoken_view() so both can be compared in one table.
    """

    # Call the tokenizer to get a BatchEncoding dict
    # It returns a BatchEncoding object (dict-like) with these keys:
    #   'input_ids'      : list of integer token IDs  ← we need this
    #   'attention_mask' : 1 for real tokens, 0 for padding  (not needed here)
    #   'token_type_ids' : segment IDs for BERT-style models (not needed here)
    batch = tok(text, add_special_tokens=False)

    # Extract the integer token ID list
    # "Hello world": [15496, 995]
    ids = batch['input_ids']

    #  Convert IDs to human-readable token pieces
    # Unlike tiktoken's decode trick, this returns the RAW internal token symbol,
    #   GPT-2  → ['Hello', 'Ġworld']    (Ġ = space before the word)
    #   LLaMA  → ['▁Hello', '▁world']   (▁ = space before the word)
    #   BERT   → ['hello', 'world']     (no marker; uses ## for sub-words)
    pieces = tok.convert_ids_to_tokens(ids)

    # Decode IDs back to a clean human-readable string ──────────────
    # tok.decode() is the reverse of encode: IDs → string.
    # NOTE: Some tokenizers apply text normalization during decode
    # (e.g., lowercasing, Unicode NFC normalization).
    # This can cause roundtrip_match to be False even when encoding succeeded.
    decoded = tok.decode(ids, skip_special_tokens=True)

    # Count and check round-trip ────────────────────────────────────
    token_count    = len(ids)
    roundtrip_match = (decoded == text)   # may be False for some tokenizers

    return {
        'pieces':          pieces,
        'ids':             ids,
        'decoded':         decoded,
        'token_count':     token_count,
        'roundtrip_match': roundtrip_match,
    }

In [7]:
def inspect_text(text):
    """
    Run all registered tokenizers on `text` and return a unified DataFrame.

    Each row corresponds to one tokenizer applied to the same input text.
    Columns: tool, text, pieces, ids, decoded, token_count, roundtrip_match

    Parameters
    ----------
    text : str — any input string to tokenize

    Returns
    -------
    pd.DataFrame — one row per tokenizer, all tokenizers on the same text
    """

    rows = []
    for name, enc in encodings.items():
        out = tiktoken_view(enc, text)
        rows.append({'tool': name, 'text': text, **out})
    for name, tok in hf_tokenizers.items():
        out = hf_view(tok, text)
        rows.append({'tool': name, 'text': text, **out})
    return pd.DataFrame(rows)


## Single-text walkthrough

In [8]:
single_text = 'Café 🚀'
inspect_text(single_text)[['tool', 'token_count', 'pieces', 'ids', 'roundtrip_match']]


,tool,token_count,pieces,ids,roundtrip_match
0,tiktoken:r50k_base,6,"[C, af, é, �, �, �]","[34, 1878, 2634, 12520, 248, 222]",True
1,tiktoken:cl100k_base,6,"[C, af, é, �, �, �]","[34, 2642, 978, 11410, 248, 222]",True
2,tiktoken:o200k_base,4,"[C, afé, �, �]","[34, 103112, 169883, 222]",True
3,hf:gpt2,6,"[C, af, Ã©, ĠðŁ, ļ, Ģ]","[34, 1878, 2634, 12520, 248, 222]",True
4,hf:llama,7,"[▁C, afé, ▁, <0xF0>, <0x9F>, <0x9A>, <0x80>]","[315, 28059, 29871, 243, 162, 157, 131]",True
5,hf:t5-small,3,"[▁Café, ▁, <unk>]","[18131, 3, 2]",False


## All examples comparison

- code,
- Hindi,
- emoji,
- punctuation-heavy text.


In [9]:
all_rows = []
for text in examples:
    all_rows.append(inspect_text(text))

results_df = pd.concat(all_rows, ignore_index=True)
results_df[['tool', 'text', 'token_count', 'roundtrip_match']]


,tool,text,token_count,roundtrip_match
0,tiktoken:r50k_base,Hello world,2,True
1,tiktoken:cl100k_base,Hello world,2,True
2,tiktoken:o200k_base,Hello world,2,True
3,hf:gpt2,Hello world,2,True
4,hf:llama,Hello world,2,True
5,hf:t5-small,Hello world,2,True
6,tiktoken:r50k_base,Café 🚀,6,True
7,tiktoken:cl100k_base,Café 🚀,6,True
8,tiktoken:o200k_base,Café 🚀,4,True
9,hf:gpt2,Café 🚀,6,True


1. Which tokenizer gives the fewest tokens for multilingual text?
2. Which tokenizers show byte-like fragments for accented characters or emoji?
3. Do token boundaries align with words, subwords, bytes, or whitespace markers?
4. Why might a code-oriented or newer tokenizer reduce token count for source code?


Code tokenization improves with newer tokenizers because:
1. Larger vocab → more code patterns absorbed as single tokens
2. Code in training data → programming keywords get dedicated entries
3. Smarter pre-tokenizer regex → operators and indentation handled better

## Average token count

This gives a coarse efficiency view across the chosen examples.

Lower is not always better in every setting, but lower token count often means lower memory usage and lower API cost for the same text.


In [10]:
avg_df = (
    results_df
    .groupby('tool', as_index=False)['token_count']
    .mean()
    .sort_values('token_count')
    .rename(columns={'token_count': 'avg_token_count'})
)
avg_df


,tool,avg_token_count
4,tiktoken:o200k_base,8.6
3,tiktoken:cl100k_base,9.4
2,hf:t5-small,9.9
0,hf:gpt2,11.0
5,tiktoken:r50k_base,11.0
1,hf:llama,11.7


## Vocabulary and tokenizer metadata

Tokenizer behavior depends on training corpus, merge rules, vocabulary size, and handling of whitespace or bytes.

This cell extracts lightweight metadata where available.


In [11]:
def safe_vocab_size(tok):
    try:
        return tok.vocab_size
    except Exception:
        try:
            return len(tok.get_vocab())
        except Exception:
            return None

meta_rows = []
for name, tok in hf_tokenizers.items():
    meta_rows.append({
        'tool': name,
        'class_name': tok.__class__.__name__,
        'vocab_size': safe_vocab_size(tok),
        'model_max_length': getattr(tok, 'model_max_length', None),
        'special_tokens': tok.all_special_tokens,
    })

for name, enc in encodings.items():
    meta_rows.append({
        'tool': name,
        'class_name': enc.__class__.__name__,
        'vocab_size': getattr(enc, 'n_vocab', None),
        'model_max_length': None,
        'special_tokens': 'encoding-defined',
    })

meta_df = pd.DataFrame(meta_rows)
meta_df


,tool,class_name,vocab_size,model_max_length,special_tokens
0,hf:gpt2,GPT2Tokenizer,50257,1024,[<|endoftext|>]
1,hf:llama,TokenizersBackend,32000,1000000000000000019884624838656,"[<s>, </s>, <unk>]"
2,hf:t5-small,T5Tokenizer,32100,512,"[</s>, <unk>, <pad>, <extra_id_0>, <extra_id_1>, <extra_id_2>, <extra_id_3>, <extra_id_4>, <extra_id_5>, <extra_id_6>, <extra_id_7>, <extra_id_8>, <extra_id_9>, <extra_id_10>, <extra_id_11>, <extr..."
3,tiktoken:r50k_base,Encoding,50257,None,encoding-defined
4,tiktoken:cl100k_base,Encoding,100277,None,encoding-defined
5,tiktoken:o200k_base,Encoding,200019,None,encoding-defined


LLaMA-2's tokenizer_config.json sets model_max_length to:
1e30 (scientific notation) — meaning "no limit defined"
Python reads this float and converts it to a huge integer

tiktoken does not store this — it is the model's responsibility, not the tokenizer's


## Error analysis view

Sometimes the interesting case is not token count but **how** text was split

In [12]:
interesting_texts = ['Café 🚀', 'नमस्ते world', '🙂🙂🙂 emoji-rich text with symbols ©™✓','def add(x, y): return x + y']
results_df[results_df['text'].isin(interesting_texts)][['tool', 'text', 'token_count', 'pieces']]


,tool,text,token_count,pieces
6,tiktoken:r50k_base,Café 🚀,6,"[C, af, é, �, �, �]"
7,tiktoken:cl100k_base,Café 🚀,6,"[C, af, é, �, �, �]"
8,tiktoken:o200k_base,Café 🚀,4,"[C, afé, �, �]"
9,hf:gpt2,Café 🚀,6,"[C, af, Ã©, ĠðŁ, ļ, Ģ]"
10,hf:llama,Café 🚀,7,"[▁C, afé, ▁, <0xF0>, <0x9F>, <0x9A>, <0x80>]"
11,hf:t5-small,Café 🚀,3,"[▁Café, ▁, <unk>]"
12,tiktoken:r50k_base,"def add(x, y): return x + y",11,"[def, add, (, x, ,, y, ):, return, x, +, y]"
13,tiktoken:cl100k_base,"def add(x, y): return x + y",10,"[def, add, (x, ,, y, ):, return, x, +, y]"
14,tiktoken:o200k_base,"def add(x, y): return x + y",10,"[def, add, (x, ,, y, ):, return, x, +, y]"
15,hf:gpt2,"def add(x, y): return x + y",11,"[def, Ġadd, (, x, ,, Ġy, ):, Ġreturn, Ġx, Ġ+, Ġy]"


#Important takeaways

- Tokenization is model-specific, not universal.
- Newer tokenizers often reduce fragmentation for multilingual text and symbols.
- Efficient tokenization can reduce both context usage and cost.
- Human-readable pieces are helpful for interpretation, but models ultimately consume IDs.
- A tokenizer that is good for English may be weak for code or non-Latin scripts.


## Reusable function

 paste any new text and inspect all tokenizers quickly


In [15]:
def compare_any_text(text, show_pieces=True):
    df = inspect_text(text)
    cols = ['tool', 'token_count', 'roundtrip_match']
    if show_pieces:
        cols.append('pieces')
    return df[cols]

compare_any_text('Large language models require careful token budgeting for long-context prompting.')


,tool,token_count,roundtrip_match,pieces
0,tiktoken:r50k_base,14,True,"[Large, language, models, require, careful, token, budget, ing, for, long, -, context, prompting, .]"
1,tiktoken:cl100k_base,13,True,"[Large, language, models, require, careful, token, budget, ing, for, long, -context, prompting, .]"
2,tiktoken:o200k_base,12,True,"[Large, language, models, require, careful, token, budgeting, for, long, -context, prompting, .]"
3,hf:gpt2,14,True,"[Large, Ġlanguage, Ġmodels, Ġrequire, Ġcareful, Ġtoken, Ġbudget, ing, Ġfor, Ġlong, -, context, Ġprompting, .]"
4,hf:llama,16,True,"[▁Lar, ge, ▁language, ▁models, ▁require, ▁careful, ▁token, ▁budget, ing, ▁for, ▁long, -, context, ▁prompt, ing, .]"
5,hf:t5-small,16,True,"[▁Large, ▁language, ▁models, ▁require, ▁careful, ▁token, ▁budget, ing, ▁for, ▁long, -, con, text, ▁prompt, ing, .]"
